# Task 2 - Incremental CPG Parser Service

The service uses Python `ast`, stable structural IDs, statement-level CFG, lexical reaching-definitions DFG, and same-file call resolution. Each file is fully released before the next file is processed.

In [1]:
from collections import Counter
from pathlib import Path
from cpg_parser.analyzer import CPGAnalyzer
from cpg_parser.discovery import discover_repo
from cpg_parser.ids import file_id

repo = Path('../source-repo')
report = discover_repo(repo)
node_counts, edge_counts = Counter(), Counter()
node_ids, edge_ids = set(), set()
warnings = 0
for relative in report.files:
    result = CPGAnalyzer(
        (repo / relative).read_text(encoding='utf-8', errors='replace'),
        file_id('huggingface/optimum', relative), relative,
    ).analyze()
    node_counts.update(result.node_counts())
    edge_counts.update(result.edge_counts())
    node_ids.update(node.id for node in result.nodes)
    edge_ids.update(edge.id for edge in result.edges)
    warnings += len(result.warnings)
print(dict(edge_counts))
print('unique node IDs:', len(node_ids) == sum(node_counts.values()))
print('unique edge IDs:', len(edge_ids) == sum(edge_counts.values()))

{
  "repository": "huggingface/optimum",
  "files": 61,
  "nodes": 62397,
  "edges": 77849,
  "node_counts": {
    "AST": 58021,
    "EXTERNAL": 2894,
    "SYNTHETIC": 1482
  },
  "edge_counts": {
    "AST": 57960,
    "CALL": 2593,
    "CFG": 7248,
    "DFG": 10048
  },
  "warnings": 31,
  "all_node_ids_unique": true,
  "all_edge_ids_unique": true
}


In [1]:
!python -m pytest

..........                                                               [100%]
10 passed in 5.24s


## Reflection

The implementation is transparent about its limits: aliases, dynamic dispatch, container mutation, and exact exception flow are conservative. A syntax error is routed to the parser-error topic instead of stopping the repository run.